# Neural Fortran Examples: Locally Connected Network on MNIST

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/cnn_mnist_1d.f90`. The file name says CNN, and the layers are locally connected layers with one-dimensional max-pooling. A locally connected layer uses a separate kernel at each position. It does not share one kernel across the image the way `conv` does.

## The network

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist_1d/app/cnn_mnist_1d.f90
:language: fortran
:lines: 1-31
```

The loader again requests training, validation, and test images. `reshape(28, 28)` makes each column a rank-2 array. In `locally_connected2d_layer` the first dimension is the channel count and the second is the width, and the output of that layer is `(filters, width)` rather than a rank-3 map. The width formula is the same valid-convolution size, $(n - 3)/s + 1$, with the default stride 1.

| Layer | Output shape |
| --- | --- |
| reshape | $28 \times 28$ |
| locally connected, 8 filters, kernel 3, ReLU | $8 \times 26$ |
| max-pool, width 2, stride 2 | $8 \times 13$ |
| locally connected, 16 filters, kernel 3, ReLU | $16 \times 11$ |
| max-pool, width 2, stride 2 | $16 \times 5$ |
| flatten, inserted before the dense layer | 80 |
| dense, softmax | 10 |

The pool calls are `maxpool(pool_width=2, stride=2)`, which selects the one-dimensional pool. The two-dimensional pool used by `cnn_mnist.f90` also passes `pool_height`. The run's `print_info` prints these shapes. A locally connected kernel is not shared across positions. The first layer stores $8 \times 26 \times 28 \times 3$ kernel values and $8 \times 26$ biases, which is 17680 parameters. The second stores $16 \times 11 \times 8 \times 3$ kernel values and $16 \times 11$ biases, which is 4400. The dense layer is $80 \times 10 + 10 = 810$. Those three counts are what this run printed.


## Training

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist_1d/app/cnn_mnist_1d.f90
:language: fortran
:lines: 33-64
```

The loop is 20 epochs, batch size 16, and SGD with learning rate 0.01, ten times the step used by `cnn_mnist.f90`. Accuracy is the `maxloc` fraction on the validation images during the loop and on the test images once at the end. The contained `accuracy` function is the same comparison as in the other two MNIST programs.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist_1d/app/cnn_mnist_1d.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist_1d/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `cnn_mnist_1d` from the release build, run in a directory that already held the six MNIST files. Standard error was empty and the process exited 0.

```{literalinclude} Fortran_Code/Section_Examples_Cnn_Mnist_1d/cnn_mnist_1d.out
:language: text
```

## What the run did

![Training curve](images/nf_cnn_mnist_1d.png)

The validation accuracy printed at the end of each epoch:

| Epoch | Validation accuracy (%) |
| ---: | ---: |
| 1 | 35.37 |
| 2 | 33.75 |
| 3 | 54.39 |
| 4 | 72.94 |
| 5 | 75.59 |
| 6 | 78.07 |
| 7 | 84.07 |
| 8 | 85.83 |
| 9 | 86.91 |
| 10 | 87.42 |
| 11 | 88.48 |
| 12 | 88.57 |
| 13 | 87.61 |
| 14 | 89.51 |
| 15 | 89.04 |
| 16 | 89.46 |
| 17 | 89.72 |
| 18 | 90.53 |
| 19 | 89.12 |
| 20 | 90.71 |

The test accuracy printed after the last epoch is 90.49 percent.


## Conclusion

The locally connected program uses the same `train` and `maxloc` accuracy as the dense MNIST example. The listing is the run from this machine.